# 🤖 AutoGen AI Workflow — Code Generation, Review & Save
### ✅ Google Colab Ready

**Workflow:**
1. 📥 User provides a query or dataset path
2. 🧠 **CodeGenerator Agent** — writes Python analysis code
3. 🔍 **CodeReviewer Agent** — audits & fixes the code
4. ⚙️ **UserProxy Agent** — executes the approved code
5. 💾 Output saved as **CSV** + **JSON** (auto-downloaded in Colab)

```
User Query / Dataset
        │
        ▼
┌──────────────────┐   review & fix   ┌──────────────────┐
│  CodeGenerator   │ ◄──────────────► │  CodeReviewer    │
│     Agent        │                  │     Agent        │
└──────────────────┘                  └──────────────────┘
        │
        ▼
┌──────────────────┐
│   UserProxy /    │
│  Executor Agent  │
└──────────────────┘
        │
        ▼
  output.csv  +  output.json
```

> **⚠️ Run cells top-to-bottom. After Step 1 you will be prompted to restart the runtime — do so, then continue from Step 2.**

---
## Step 1 · Install Dependencies
> After this cell finishes, **Runtime → Restart session**, then continue from Step 2.

In [ ]:
# ── Install AutoGen (correct package name) + extras ──────────────────────────
# pyautogen is the pip package; it exposes `autogen` as the import name.
# Pin to >=0.2 which includes GroupChat & GroupChatManager.
import subprocess, sys

packages = [
    "pyautogen>=0.2.0",
    "openai>=1.0.0",
    "pandas",
    "termcolor",   # pretty agent output
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("\n✅ All packages installed.")
print("⚠️  Now go to Runtime → Restart session, then run cells from Step 2 onwards.")

---
## Step 2 · Verify Imports
> **Start here after restarting the runtime.**

In [ ]:
# ── Verify that autogen imported correctly ───────────────────────────────────
try:
    import autogen
    print(f"✅ autogen imported  — version: {autogen.__version__}")
except ModuleNotFoundError:
    raise SystemExit(
        "❌ autogen not found.\n"
        "   Did you restart the runtime after Step 1?\n"
        "   If yes, re-run Step 1 then restart again."
    )

import pandas as pd
import json, os, re
from pathlib import Path

# ── Detect whether we are running inside Google Colab ────────────────────────
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("ℹ️  Running outside Colab (local Jupyter)")

---
## Step 3 · (Optional) Mount Google Drive
Skip this step if you do not want to persist outputs to Drive.

In [ ]:
MOUNT_DRIVE = False   # ← Set to True to save outputs to your Google Drive

if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = Path("/content/drive/MyDrive/autogen_outputs")
    print("✅ Drive mounted. Outputs will be saved to Google Drive.")
else:
    WORKSPACE = Path("/content/autogen_workspace") if IN_COLAB else Path("autogen_workspace")
    print(f"ℹ️  Outputs will be saved to: {WORKSPACE}")

WORKSPACE.mkdir(parents=True, exist_ok=True)

---
## Step 4 · LLM Configuration
> Paste your **OpenAI API key** below. You can also use Azure OpenAI or any OpenAI-compatible endpoint.

In [ ]:
# ── Option A: Hard-code key (simple, less secure) ────────────────────────────
OPENAI_API_KEY = "sk-..."          # ← Replace with your key

# ── Option B: Use Colab Secrets (recommended) ────────────────────────────────
# 1. Click the 🔑 icon in the left sidebar → Add secret → name: OPENAI_API_KEY
# 2. Uncomment the 3 lines below and comment out the line above.
# from google.colab import userdata
# OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
# print("✅ Key loaded from Colab Secrets")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

LLM_CONFIG = {
    "config_list": [
        {
            "model": "gpt-4o",              # swap to gpt-4o-mini to save quota
            "api_key": OPENAI_API_KEY,
        }
    ],
    "temperature": 0.2,
    "cache_seed": 42,
}

print("✅ LLM config ready  (model: gpt-4o)")

---
## Step 5 · Define the Three Agents

In [ ]:
# ── Agent 1 · CodeGenerator ───────────────────────────────────────────────────
code_generator = autogen.AssistantAgent(
    name="CodeGenerator",
    llm_config=LLM_CONFIG,
    system_message=(
        "You are an expert Python data scientist.\n"
        "When given a user query or dataset description, write clean, "
        "well-commented Python code that:\n"
        "  1. Loads or generates the data (use pandas).\n"
        "  2. Performs the requested analysis / transformation.\n"
        "  3. Saves the final result as BOTH 'output.csv' AND 'output.json' "
        "in the current working directory.\n"
        "Wrap ALL executable code in a single ```python ... ``` block.\n"
        "After writing the code, say TERMINATE."
    ),
)

# ── Agent 2 · CodeReviewer ───────────────────────────────────────────────────
code_reviewer = autogen.AssistantAgent(
    name="CodeReviewer",
    llm_config=LLM_CONFIG,
    system_message=(
        "You are a senior Python code reviewer.\n"
        "When CodeGenerator provides code:\n"
        "  1. Check for correctness, missing imports, and edge cases.\n"
        "  2. Confirm the code saves output BOTH as 'output.csv' AND 'output.json'.\n"
        "  3. If correct → reply: LGTM. TERMINATE\n"
        "  4. If issues found → provide the fully corrected code in a single "
        "```python ... ``` block, then say TERMINATE."
    ),
)

# ── Agent 3 · UserProxy / Executor ──────────────────────────────────────────
user_proxy = autogen.UserProxyAgent(
    name="UserProxy",
    human_input_mode="NEVER",           # fully automated
    max_consecutive_auto_reply=10,
    code_execution_config={
        "work_dir": str(WORKSPACE),     # executes code here
        "use_docker": False,            # Docker not available in Colab
    },
    is_termination_msg=lambda msg: "TERMINATE" in (msg.get("content") or ""),
)

print("✅ Agents ready: CodeGenerator · CodeReviewer · UserProxy")

---
## Step 6 · Helper: Save Results

In [ ]:
def save_results(data, filename_stem="output"):
    """
    Save a DataFrame / list / dict as both CSV and JSON.
    Returns (csv_path, json_path).
    """
    csv_path  = WORKSPACE / f"{filename_stem}.csv"
    json_path = WORKSPACE / f"{filename_stem}.json"

    if isinstance(data, pd.DataFrame):
        df = data
    elif isinstance(data, list):
        df = pd.DataFrame(data)
    elif isinstance(data, dict):
        df = pd.DataFrame([data])
    else:
        raise TypeError(f"Unsupported type: {type(data)}")

    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient="records", indent=2)

    print(f"✅ CSV  saved → {csv_path}")
    print(f"✅ JSON saved → {json_path}")
    return str(csv_path), str(json_path)


print("✅ save_results() helper ready.")

---
## Step 7 · Run the AutoGen Workflow

In [ ]:
def run_autogen_workflow(user_query: str) -> dict:
    """
    Orchestrate: UserProxy → CodeGenerator → CodeReviewer → Execute → Save.
    Returns dict with keys 'csv' and/or 'json' pointing to output files.
    """
    print("=" * 65)
    print("🚀  AutoGen Workflow Starting")
    print("=" * 65)
    print(f"📝 Query:\n{user_query.strip()}\n")

    groupchat = autogen.GroupChat(
        agents=[user_proxy, code_generator, code_reviewer],
        messages=[],
        max_round=12,
        speaker_selection_method="round_robin",
    )
    manager = autogen.GroupChatManager(
        groupchat=groupchat,
        llm_config=LLM_CONFIG,
    )

    user_proxy.initiate_chat(
        manager,
        message=(
            f"{user_query.strip()}\n\n"
            f"Save the result as BOTH 'output.csv' AND 'output.json' "
            f"in the directory: {WORKSPACE}"
        ),
    )

    # ── Check output files ────────────────────────────────────────
    print("\n" + "=" * 65)
    print("📂  Output file check")
    results = {}
    for key, fname in [("csv", "output.csv"), ("json", "output.json")]:
        p = WORKSPACE / fname
        if p.exists():
            results[key] = str(p)
            print(f"   ✅ {fname} → {p}")
        else:
            print(f"   ⚠️  {fname} not found")

    # ── Auto-download in Colab ────────────────────────────────────
    if IN_COLAB and results:
        from google.colab import files
        print("\n📥  Downloading output files to your computer...")
        for path in results.values():
            files.download(path)

    return results


print("✅ run_autogen_workflow() ready.")

---
## Step 8 · Enter Your Query & Run

Edit **`USER_QUERY`** below — use plain English to describe what you want.

In [ ]:
# ── ✏️  Edit your query here ──────────────────────────────────────────────────
USER_QUERY = """
Generate a synthetic e-commerce sales dataset with 100 rows and columns:
  product_name, category, units_sold, unit_price, revenue, sale_date.
Then calculate total revenue per category and save the summary table.
"""

# ── Example 2: analyse an uploaded / existing CSV (uncomment to use) ─────────
# USER_QUERY = """
# Load '/content/my_data.csv', compute descriptive statistics for all
# numeric columns, find the top-5 rows by the first numeric column,
# and save the results.
# """

# ─────────────────────────────────────────────────────────────────────────────
output_files = run_autogen_workflow(USER_QUERY)

---
## Step 9 · Preview Outputs

In [ ]:
# ── CSV preview ───────────────────────────────────────────────────────────────
if "csv" in output_files:
    df_out = pd.read_csv(output_files["csv"])
    print(f"📊 CSV shape: {df_out.shape}")
    display(df_out.head(10))
else:
    print("No CSV output found.")

In [ ]:
# ── JSON preview ──────────────────────────────────────────────────────────────
if "json" in output_files:
    with open(output_files["json"]) as f:
        records = json.load(f)
    print(f"📋 JSON records (first 5 of {len(records)}):")
    print(json.dumps(records[:5], indent=2))
else:
    print("No JSON output found.")

---
## Step 10 · Inspect the Final Generated Code

In [ ]:
def extract_last_code_block(messages):
    """Return the last Python code block from chat history."""
    pattern = re.compile(r"```python\n(.*?)```", re.DOTALL)
    for msg in reversed(messages):
        content = msg.get("content") or ""
        hits = pattern.findall(content)
        if hits:
            return hits[-1].strip()
    return "No code block found."

try:
    final_code = extract_last_code_block(groupchat.messages)
    print("📝 Final reviewed & executed code:\n")
    print(final_code)
except NameError:
    print("Run Step 8 first to populate the chat history.")

---
## Step 11 · (Optional) Bring Your Own Data

In [ ]:
# ── Upload a file in Colab, then reference it in USER_QUERY ──────────────────
if IN_COLAB:
    from google.colab import files
    print("Click 'Choose Files' to upload a CSV or JSON dataset.")
    uploaded = files.upload()
    for filename in uploaded:
        print(f"✅ Uploaded: /content/{filename}")
        print(f"   Use this path in USER_QUERY: '/content/{filename}'")
else:
    print("Place your file in the working directory and reference it in USER_QUERY.")

---
## Quick Reference

| Step | Who | What happens |
|------|-----|--------------|
| 1 | You | Install packages → **restart runtime** |
| 2–4 | Setup | Imports, Drive mount, API key |
| 5 | AutoGen | Three agents defined |
| 6 | Helper | `save_results()` utility |
| 7 | AutoGen | GroupChat wired up |
| **8** | **You** | **Edit query → Run workflow** |
| 9 | Output | CSV & JSON previewed in notebook |
| 9 | Colab | Files auto-downloaded to your computer |

**Troubleshooting**
- `No module named 'autogen'` → Re-run Step 1 and **restart runtime** before Step 2.
- `AuthenticationError` → Check your OpenAI API key in Step 4.
- `output.csv not found` → Increase `max_round` in `run_autogen_workflow` or simplify the query.
- **Model swap**: change `"gpt-4o"` to `"gpt-4o-mini"` in Step 4 for lower cost.